In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import *


import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


from sklearn.model_selection import train_test_split

from collections import Counter

import os, sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import tensorflow as tf

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Conv1D, GlobalMaxPooling1D, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.initializers import glorot_normal
from tensorflow.keras.activations import sigmoid
from sklearn.metrics import roc_auc_score
import utils
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping

#from nettcr_architectures import nettcr_ab, nettcr_one_chain

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU') 
tf.config.experimental.set_memory_growth(physical_devices[0], True)

In [ ]:
# ### load data




# train_data= pd.read_csv('1_MCPAS_final_datasets/1_origMCPAS_NoMHC/1_origMCPAS_noMHC_train_AB.csv')

# test_data = pd.read_csv('1_MCPAS_final_datasets/1_origMCPAS_NoMHC/4_origMCPAS_noMHC_test_AB.csv')



# train_data= pd.read_csv('2_VDJDB_final_datasets/1_origVDJDB_NoMHC/2_origVDJDB_NoMHC_train_B.csv')

# test_data = pd.read_csv('2_VDJDB_final_datasets/1_origVDJDB_NoMHC/5_origVDJDB_NoMHC_test_B.csv')


#### ext test data case PTMNET
##PTMNET data

#test_data = pd.read_csv('4_pMTnet_test/test6190_set1.csv')

#test_data = pd.read_csv('4_pMTnet_test/test6190_set2.csv')

#test_data = pd.read_csv('4_pMTnet_test/test6190_set3.csv')

#test_data = pd.read_csv('4_pMTnet_test/test6190_set4.csv')

#test_data = pd.read_csv('4_pMTnet_test/test6190_set5.csv')

#test_data = pd.read_csv('4_pMTnet_test/test6190_set6.csv')

#test_data = pd.read_csv('4_pMTnet_test/test6190_set7.csv')

#test_data = pd.read_csv('4_pMTnet_test/test6190_set8.csv')

In [ ]:
train_data

In [ ]:
test_data

In [ ]:
train_data.shape, test_data.shape


In [ ]:
max_b, max_pep = max([len(item) for item in train_data.tcrb.values.tolist()]), max([len(item) for item in train_data.peptide.values.tolist()])

In [ ]:
max_b, max_pep

In [ ]:
max([len(item) for item in test_data.tcrb.values.tolist()]), max([len(item) for item in test_data.peptide.values.tolist()])

In [ ]:
# max([len(item) for item in test_data.tcrb.values.tolist()]), max([len(item) for item in test_data.peptide.values.tolist()])

In [ ]:
chain = "ab"#["a","b","ab"]

In [ ]:
# # train_data = data_train#pd.read_csv(args.trainfile)
# # test_data = data_test#pd.read_csv(args.testfile) might not be useful here

# train_data = sample_train
# test_data = sample_test

In [ ]:
# Encode data
encoding = utils.blosum50_20aa
#early_stop = EarlyStopping(monitor='loss',min_delta=0,patience=10, verbose=0,mode='min',restore_best_weights=True)

In [ ]:
def clear_sess():
  try:
    del model 
    del history
  except:
    pass
  from tensorflow.keras import backend as K
  K.clear_session()
  import gc
  gc.collect()
  return None

In [ ]:
clear_sess()

In [ ]:
def keras_mcc(y_true, y_pred):
    tp = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    tn = K.sum(K.round(K.clip((1 - y_true) * (1 - y_pred), 0, 1)))
    fp = K.sum(K.round(K.clip((1 - y_true) * y_pred, 0, 1)))
    fn = K.sum(K.round(K.clip(y_true * (1 - y_pred), 0, 1)))

    num = tp * tn - fp * fn
    den = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / K.sqrt(den + K.epsilon())

In [ ]:
### model

def nettcr_one_chain(max_b, max_pep):
    cdr_in = Input(shape=(max_b,20)) ### prev (30,20)
    pep_in = Input(shape=(max_pep,20)) ### prev (9,20)
    
    pep_conv1 = Conv1D(16, 1, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(pep_in)
    pep_pool1 = GlobalMaxPooling1D()(pep_conv1)
    pep_conv3 = Conv1D(16, 3, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(pep_in)
    pep_pool3 = GlobalMaxPooling1D()(pep_conv3)
    pep_conv5 = Conv1D(16, 5, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(pep_in)
    pep_pool5 = GlobalMaxPooling1D()(pep_conv5)
    pep_conv7 = Conv1D(16, 7, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(pep_in)
    pep_pool7 = GlobalMaxPooling1D()(pep_conv7)
    pep_conv9 = Conv1D(16, 9, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(pep_in)
    pep_pool9 = GlobalMaxPooling1D()(pep_conv9)
    
    cdr_conv1 = Conv1D(16, 1, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(cdr_in)
    cdr_pool1 = GlobalMaxPooling1D()(cdr_conv1)
    cdr_conv3 = Conv1D(16, 3, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(cdr_in)
    cdr_pool3 = GlobalMaxPooling1D()(cdr_conv3)
    cdr_conv5 = Conv1D(16, 5, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(cdr_in)
    cdr_pool5 = GlobalMaxPooling1D()(cdr_conv5)
    cdr_conv7 = Conv1D(16, 7, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(cdr_in)
    cdr_pool7 = GlobalMaxPooling1D()(cdr_conv7)
    cdr_conv9 = Conv1D(16, 9, padding='same', activation='sigmoid', kernel_initializer='glorot_normal')(cdr_in)
    cdr_pool9 = GlobalMaxPooling1D()(cdr_conv9)
    
    pep_cat = concatenate([pep_pool1, pep_pool3, pep_pool5, pep_pool7, pep_pool9])
    cdr_cat = concatenate([cdr_pool1, cdr_pool3, cdr_pool5, cdr_pool7, cdr_pool9])
    
    cat = concatenate([pep_cat, cdr_cat], axis=1)
    
    dense = Dense(256, activation='sigmoid')(cat)
    
    #drop = Dropout(0.3)(dense)
    
    out = Dense(1, activation='sigmoid')(dense)
    
    model = (Model(inputs=[cdr_in, pep_in],outputs=[out]))
    
    return model

In [ ]:
# Call and compile the model ### test not needed-checking cross validation
if chain=='ab':
    pep_train  = utils.enc_list_bl_max_len(train_data.peptide, encoding, 25)
    tcra_train = utils.enc_list_bl_max_len(train_data.tcra, encoding, 30)
    tcrb_train = utils.enc_list_bl_max_len(train_data.tcrb, encoding, 30)
    y_train    = np.array(train_data.sign).reshape(-1,1)

#     pep_test   = utils.enc_list_bl_max_len(test_data.peptide, encoding, 25)
#     tcra_test  = utils.enc_list_bl_max_len(test_data.tcra, encoding, 30)
#     tcrb_test  = utils.enc_list_bl_max_len(test_data.tcrb, encoding, 30)
#     y_test     = np.array(test_data.sign).reshape(-1,1)
    
    ##### test_train_split not needed for CV
   
    
    tcra_train, tcra_test, tcrb_train, tcrb_test, pep_train, pep_test, y_train, y_test =  train_test_split(tcra_train, tcrb_train, pep_train, y_train, test_size=0.20, random_state=split_seed)
    ##### 
    
    train_inputs = [tcra_train, tcrb_train, pep_train]
    test_inputs  = [tcra_test, tcrb_test, pep_test]
    
    print(tcra_train.shape, tcra_test.shape, tcrb_train.shape, tcrb_test.shape, pep_train.shape, pep_test.shape, y_train.shape, y_test.shape)

    model = nettcr_ab()
    
elif chain=="a":
    pep_train  = utils.enc_list_bl_max_len(train_data.peptide, encoding, 25)
    tcra_train = utils.enc_list_bl_max_len(train_data.tcra, encoding, 30)
    y_train    = np.array(train_data.sign).reshape(-1,1)

#     pep_test   = utils.enc_list_bl_max_len(test_data.peptide, encoding, 25)
#     tcra_test  = utils.enc_list_bl_max_len(test_data.tcra, encoding, 30)
#     y_test     = np.array(test_data.sign).reshape(-1,1)
    
    ##### test_train_split not needed for CV
    
    tcra_train, tcra_test, pep_train, pep_test, y_train, y_test =  train_test_split(tcra_train, pep_train, y_train, test_size=0.20, random_state=split_seed)
    
    
    train_inputs = [tcra_train, pep_train]
    test_inputs = [tcra_test, pep_test]
                                                                                    
    print(tcra_train.shape, tcra_test.shape, pep_train.shape, pep_test.shape, y_train.shape, y_test.shape)                                                                         
                                                                                    
                                                                                    
    model = nettcr_one_chain()
    
    
elif chain=="b":
    
    
    pep_train  = utils.enc_list_bl_max_len(train_data.peptide, encoding, max_pep)
    tcrb_train = utils.enc_list_bl_max_len(train_data.tcrb, encoding, max_b)
    y_train    = np.array(train_data.sign).reshape(-1,1)

    pep_test   = utils.enc_list_bl_max_len(test_data.Antigen, encoding, max_pep)
    tcrb_test  = utils.enc_list_bl_max_len(test_data.CDR3, encoding, max_b)
    y_test     = np.array(test_data.label).reshape(-1,1)
    
    
    
#     ### 000
#     pep_test   = utils.enc_list_bl_max_len(test_data.peptide, encoding, 20)
#     tcrb_test  = utils.enc_list_bl_max_len(test_data.tcrb, encoding, 38)
#     y_test     = np.array(test_data.sign).reshape(-1,1)
    
    ##### test_train_split not needed for CV
    
    #tcrb_train, tcrb_test, pep_train, pep_test, y_train, y_test =  train_test_split(tcrb_train, pep_train, y_train, test_size=0.20, random_state=split_seed)

    
    train_inputs = [tcrb_train, pep_train]
    test_inputs = [tcrb_test, pep_test]
                                                                                    
                                                                                    
    print(tcrb_train.shape, tcrb_test.shape, pep_train.shape, pep_test.shape, y_train.shape, y_test.shape)                                                                                
                                                                                    
                                                                                    
                                                                                    
    model = nettcr_one_chain(max_b, max_pep)

In [ ]:
# mdl.summary()

In [ ]:
metrics_c = [#'accuracy',
    #keras.metrics.FalseNegatives(name="fn"),
    #keras.metrics.FalsePositives(name="fp"),
    #keras.metrics.TrueNegatives(name="tn"),
    #keras.metrics.TruePositives(name="tp"),
    #keras.metrics.BinaryAccuracy(name='accuracy'),
    #tensorflow.keras.metrics.Precision(name="precision"),
    #tensorflow.keras.metrics.Recall(name="recall"),
    tf.keras.metrics.AUC(name="auc_pr",curve="PR"),
    tf.keras.metrics.AUC(name="auc_roc",curve="ROC")
]

In [ ]:
model.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=0.001), metrics=metrics_c) #0.001 prev

In [ ]:
early_stop = EarlyStopping(monitor='loss',min_delta=0,
               patience=10, verbose=0,mode='min',restore_best_weights=True)

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.7,patience=10, min_lr=0.000001)

In [ ]:
checkpoint_filepath = 'weights-improvement.hdf5'
model_checkpoint_callback = ModelCheckpoint(filepath=checkpoint_filepath,save_weights_only=False,monitor='val_auc_roc',mode='max',save_best_only=True)

In [ ]:
history = model.fit(train_inputs, y_train,epochs=100,batch_size=256, verbose=1, 
                  #validation_split=0.2, 
                  validation_data=([test_inputs],y_test),
                  callbacks=[reduce_lr,
                             model_checkpoint_callback
                            ])
print('done')

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn')
%matplotlib inline

In [ ]:
#@title Classification Hidden
#training plots ##Classification
fig = plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Classification Losses")
plt.ylabel("Classification Losses")
plt.xlabel("Epoch")
plt.legend(["Training Classification Loss","Valid Classification Loss"])

plt.subplot(1,3,2)
plt.plot(history.history['auc_pr'])
plt.plot(history.history['val_auc_pr'])
plt.title("Classification AUC PR")
plt.ylabel("Classification AUC PR")
plt.xlabel("Epoch")
plt.legend(["Training AUC PR","Valid AUC PR"])

plt.subplot(1,3,3)
plt.plot(history.history['auc_roc'])
plt.plot(history.history['val_auc_roc'])
plt.title("Classification AUC ROC")
plt.ylabel("Classification AUC ROC")
plt.xlabel("Epoch")
plt.legend(["Training AUC ROC","Valid AUC ROC"])

plt.show()

In [ ]:
####model load
#model_loaded = 'weights-improvement-0.6.hdf5'
model_loaded = 'weights-improvement.hdf5'
model = tf.keras.models.load_model(model_loaded,compile=True)

In [ ]:
y_pred = model.predict([test_inputs])

In [ ]:
y_act = y_test.flatten()

In [ ]:
y_pred= y_pred.flatten()

In [ ]:
y_pred_c=np.where(y_pred>0.5,1,0)

In [ ]:
confusion_matrix(y_act,y_pred_c )

In [ ]:
print(classification_report(y_act,y_pred_c))

In [ ]:
print('AUC-ROC', roc_auc_score(y_act, y_pred))
print('AUC-PR',  average_precision_score(y_act, y_pred))

print('MCC', matthews_corrcoef(y_act,y_pred_c))
print('CKS', cohen_kappa_score(y_act,y_pred_c))

In [ ]:
print(roc_auc_score(y_act, y_pred),average_precision_score(y_act, y_pred),matthews_corrcoef(y_act,y_pred_c),cohen_kappa_score(y_act,y_pred_c))